# Exploratory Data Analysis and unsupervised learning

for Body mass prediction in biped dinosaurs using supervised learning
-----------------------------------------------------------------------------------------------------------------------------
Alejandro Otero - CONICET - Museo de La Plata (UNLP), Argentina. alexandros.otero@gmail.com

Ailen Borya - Especialización en Geociencias de Exploración y Desarrollo de Hidrocarburos (UNLP), Argentina. ailenborya@gmail.com

## 1. Libraries

In [ ]:
# Preproccesing and EDA
import os
import pandas as pd
import math
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Cluster(K-Means)
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn import preprocessing
from sklearn.preprocessing import scale

## 2. Open file

In [ ]:
# Open dataset with Google colab

from google.colab import files
uploaded = files.upload()
data = pd.read_excel("train_data.xlsx")
data

In [ ]:
# Open file 

data = pd.read_excel('train_data.xlsx')
data.head(5) # Visualising first 5 rows

## 3. EDA

In [ ]:
# Show the column names
data.columns

In [ ]:
# Show the non-null values count and data type per column
data.info()

In [ ]:
# Show the range of values per column and the statistical parameters per column

data.describe()

In [ ]:
#Export describe as excel file

describe=data.describe().to_excel("Table S5.xlsx", index=True)

### 3.1. Visualize data distribution

In [ ]:
#Create new datasets

df_long = data.melt(
    id_vars="State",
    value_vars=['Femur length (mm)',
                'Femur circumference (mm)'],
    var_name="Predictors",
    value_name="Value"
)

df_bm = data.melt(
    id_vars="State",
    value_vars=['Body mass (g)'],
    var_name="Target",
    value_name="Value_bm"
)
#Merge the datasets

df_merge=pd.merge(df_long,
                  df_bm[['State',
                         'Target',
                         "Value_bm"]],
                  on="State",
                  how="left"
                  )
df_merge

In [ ]:
# Define subplots

fig = make_subplots(
    rows = 1,
    cols = 2,
    vertical_spacing = 0.1,
    subplot_titles = (
        "<span style='font-size: 18px;'><b>Statistical parameters of predictors<b></span>",
        "<span style='font-size: 18px;'><b>Statistical parameters of target<b></span>"
    ),
)

# Subplot 1
fig.add_trace(
    go.Box(
        x = df_merge['Predictors'],
        y = df_merge['Value']
    ),
    row = 1, col = 1

)# Subplot 2
fig.add_trace(
    go.Box(
        x =df_merge['Target'],
        y = df_merge['Value_bm']
    ),
    row = 1, col = 2
)


# Set y axes log scale
fig.update_yaxes(
    type = 'log',
    row = 1, col = 1
)

fig.update_yaxes(
    type = 'log',
    row = 1, col = 2
)
# Set graph size
fig.update_layout(
    height = 600,
    width = 1200
)

fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor= "lightgray"
)
# Edit axis labels
fig['layout']['xaxis']['title'] = "<span style='font-size: 16px;'><b>Predictors<b></span>"
fig['layout']['xaxis2']['title'] = "<span style='font-size: 16px;'><b>Target<b></span>"
fig['layout']['yaxis']['title'] = "<span style='font-size: 16px;'><b>Values (log)<b></span>"
fig['layout']['yaxis2']['title'] = "<span style='font-size: 16px;'><b>Values (log)<b></span>"

# Delete the legend
fig.update_layout(showlegend=False)

#Show the boxplots
fig.show()


In [ ]:
#Create a copy of the dataset

df = data.copy()

In [ ]:
# Drop the Species and Source columns

df.drop(columns=['Species','Source'])

In [ ]:
#Visual method- Normal distribution

sns.histplot(data['Body mass (g)'],
             log_scale=True,
             color="cornflowerblue",
             kde=True)

plt.title('Body mass distribution', fontsize=14)
plt.xlabel("Body mass (log)", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)

# Print and save as .jpeg file
plt.savefig('Bm_histogram.jpeg',dpi=500,bbox_inches="tight")


In [ ]:
# Set Higher clade colors

color_dict={
    "Anseriformes":"rgba(230,192,186,1)",
    "Charadriiformes": "rgba(217,141,126,1)",
    "Psittaciformes": "rgba(205,91,67,1)",
    "Caprimulgiformes": "rgba(245,142,118,1)",
    "Passeriformes": "rgba(229,201,20,1)",
    "Gruiformes": "rgba(145,255,175,1)",
    "Accipitriformes": "rgba(114,201,145,1)",
    "Coraciiformes": "rgba(35,158,96,1)",
    "Apodiformes": "rgba(224,205,226,1)",
    "Falconiformes": "rgba(185,150,194,1)",
    "Columbiformes": "rgba(140,74,154,1)",
    "Suliformes": "rgba(205,88,151,0.9)",
    "Podicipediformes": "rgba(156,185,249,1)",
    "Strigiformes": "rgba(106,142,221,1)",
    "Procellariiformes":"rgba(71,74,71,1)" ,
    "Ciconiiformes": "rgba(158,158,158,1)",
    "Galliformes":"rgba(197,197,197,1)",
    "Trogoniformes": "rgba(143,153,142,1)",
    "Cuculiformes": "rgba(215,204,17,1)",
    "Tinamiformes": "rgba(202,144,86,1)",
    "Struthioniformes": "rgba(246,170,67,1)",
    "Pelecaniformes": "rgba(218, 156, 133,1)",
    "Sphenisciformes": "rgba(50,89,18,1)",
    "Phoenicopteriformes": "rgba(32,130,110,1)",
    "Theropoda":  "rgba(92,188,93,1)"
}

In [ ]:
# Body mass according to Higher clade

fig = px.box(data, x='Higher clade',
             y='Body mass (g)',
             log_y= True,
             color= 'Higher clade',
             color_discrete_map=color_dict,
             points="outliers",
             title=("<span style='font-size: 18px;'><b>Body mass according to Higher clade<b></span>"
             ),
             width=1100,
             height=600)

fig.update_layout(
    plot_bgcolor="white",  # Inner area
    paper_bgcolor= "lightgray"  # Outer area
)

fig.add_hline(
    y = 10000,
    line_width = 1,
    line_dash = 'dash',
    line_color = 'black'
)

fig['layout']['xaxis']['title'] = "<span style='font-size: 16px;'><b>Higher clade<b></span>"
fig['layout']['yaxis']['title'] = "<span style='font-size: 16px;'><b>Body mass (g)<b></span>"

# Delete the legend
fig.update_layout(showlegend=False)
fig.update_xaxes(tickfont=dict(size=14))
fig.update_yaxes(tickfont=dict(size=14))
fig.show()

In [ ]:
# Body mass according to Higher clade
# Show the statistical parameters per higher clade

body_mass=df.groupby('Higher clade')['Body mass (g)']
body_mass.describe()


In [ ]:
# Plotting the record according to Higher clade

sizes = body_mass.size()
sizes = sizes.sort_values(ascending=False) # Sort counts in descending order

plt.figure(figsize=(8, 6))
bars = plt.bar(
    sizes.index,
    sizes.values,
    color="cornflowerblue"
)

# Add labels on bars
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, yval, round(yval, 2),
             ha='center', va='bottom', fontsize=10)

plt.xticks(rotation=45, ha='right',fontsize=10)
plt.yticks(fontsize=10)
plt.rc('axes', labelsize=12)
plt.ylabel("Count of records", fontsize=12)
plt.title("Records according to Higher clade", fontsize=14)
plt.tight_layout()

# Print and save as .jpeg file
plt.savefig('HigherClade_bar.jpeg',dpi=500,bbox_inches="tight")

In [ ]:
# Body mass according to State (extant and extinct)

fig = px.box(
    data,
    x='State',
    y='Body mass (g)',
    log_y= True,
    color= 'State',
    points="outliers",
    title=("<span style='font-size: 18px;'><b>Body mass according to State<b></span>"
    ),
    width=500,
    height=500
    )

fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor= "lightgray"
)

fig.add_hline(
    y = 10000,
    line_width = 1,
    line_dash = 'dash',
    line_color = 'black'
)

fig['layout']['xaxis']['title'] = "<span style='font-size: 16px;'><b>State<b></span>"
fig['layout']['yaxis']['title'] = "<span style='font-size: 16px;'><b>Body mass (g)<b></span>"

fig.update_layout(showlegend=False)
fig.update_xaxes(tickfont=dict(size=14))
fig.update_yaxes(tickfont=dict(size=14))
fig.show()



In [ ]:
# Body mass according to State
# Show the statistical parameters per state

state=df.groupby("State")['Body mass (g)']
state.describe()

In [ ]:
# Plotting the record according to state

s_sizes= state.size()
s_sizes= s_sizes.sort_values(ascending=False) # Sort counts in descending order

plt.figure(figsize=(5, 4))
bars = plt.bar(
    s_sizes.index,
    s_sizes.values,
    width=0.3,
    color=["cornflowerblue","lightcoral"]
)

# Add labels on bars
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, yval, round(yval, 2),
             ha='center', va='bottom', fontsize=10)

plt.xticks(rotation=45, ha='right',fontsize=10)
plt.yticks(fontsize=10)
plt.ylabel("Count of records", fontsize=12)
plt.title("Records according to State", fontsize=14)
plt.tight_layout()

# Print and save as .jpeg file
plt.savefig('State_bar.jpeg',dpi=500,bbox_inches="tight")

#### Femur Lenght variable (predictor)

In [ ]:
#Visual method- Normal distribution

sns.histplot(data['Femur length (mm)'],
             color="cornflowerblue",
             kde=True)

plt.title('Femur lenght distribution', fontsize=14)
plt.rc('axes', labelsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.show()

In [ ]:
# Femur lenght according to Higher clade

fig = px.box(data, x='Higher clade',
             y='Femur length (mm)',
             log_y= True,
             color= 'Higher clade',
             color_discrete_map=color_dict,
             points="outliers",
             title=("<span style='font-size: 18px;'><b>Femur lenght according to Higher clade<b></span>"
             ),
             width=1100,
             height=600
             )

fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor= "lightgray"
)

fig.add_hline(
    y = 200,
    line_width = 1,
    line_dash = 'dash',
    line_color = 'black'
)

fig['layout']['xaxis']['title'] = "<span style='font-size: 16px;'><b>Higher clade<b></span>"
fig['layout']['yaxis']['title'] = "<span style='font-size: 16px;'><b>Femur lenght (mm)<b></span>"

fig.update_layout(showlegend=False)
fig.update_xaxes(tickfont=dict(size=14))
fig.update_yaxes(tickfont=dict(size=14))
fig.show()

In [ ]:
# Femur lenght according to Higher clade
# Show the statistical parameters per higher clade

femur_length=df.groupby('Higher clade')['Femur length (mm)']
femur_length.describe()

In [ ]:
# Femur lenght according to State (extant and extinct)

fig = px.box(data, x='State',
             y='Femur length (mm)',
             log_y= True,
             color= 'State',
             points="outliers",
             title=("<span style='font-size: 18px;'><b>Femur lenght according to State<b></span>"
             ),
             width=500,
             height=500)

fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor= "lightgray"
)

fig.add_hline(
    y = 200,
    line_width = 1,
    line_dash = 'dash',
    line_color = 'black'
)

fig['layout']['xaxis']['title'] = "<span style='font-size: 16px;'><b>State<b></span>"
fig['layout']['yaxis']['title'] = "<span style='font-size: 16px;'><b>Femur lenght (mm)<b></span>"

fig.update_layout(showlegend=False)
fig.update_xaxes(tickfont=dict(size=14))
fig.update_yaxes(tickfont=dict(size=14))
fig.show()

In [ ]:
# Femur_length_mm according to State
# Show the statistical parameters per state

Femur_length_state=df.groupby("State")['Femur length (mm)']
Femur_length_state.describe()

#### Femur Circunference variable (predictor)

In [ ]:
#Visual method- Normal distribution

sns.histplot(data['Femur circumference (mm)'],
             color="cornflowerblue",
             kde=True)
plt.title('Femur circumference distribution', fontsize=14)
plt.rc('axes', labelsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.show()



In [ ]:
# Femur Circumference to higher clade

fig = px.box(data,
             x='Higher clade',
             y='Femur circumference (mm)',
             log_y= True,
             color= 'Higher clade',
             color_discrete_map=color_dict,
             points="outliers",
             title=("<span style='font-size: 18px;'><b>Femur circumference according to Higher clade<b></span>"
             ),
             width=1100,
             height=600)

fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor= "lightgray"
)

fig.add_hline(
    y = 70,
    line_width = 1,
    line_dash = 'dash',
    line_color = 'black'
)

fig['layout']['xaxis']['title'] = "<span style='font-size: 16px;'><b>Higher clade<b></span>"
fig['layout']['yaxis']['title'] = "<span style='font-size: 16px;'><b>Femur circumference (mm)<b></span>"

fig.update_layout(showlegend=False)
fig.update_xaxes(tickfont=dict(size=14))
fig.update_yaxes(tickfont=dict(size=14))
fig.show()

In [ ]:
# Femur circumference according to Higher clade
# Show the statistical parameters per higher clade

femur_circumference=df.groupby('Higher clade')['Femur circumference (mm)']
femur_circumference.describe()


In [ ]:
# Femur Circunference to State

fig = px.box(data, x='State',
             y='Femur circumference (mm)',
             log_y= True, color= 'State',
             points="outliers",
             title=("<span style='font-size: 18px;'><b>Femur circumference according to State<b></span>"
             ),
             width=500,
             height=500)

fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor= "lightgray"
)

fig.add_hline(
    y = 70,
    line_width = 1,
    line_dash = 'dash',
    line_color = 'black'
)

fig['layout']['xaxis']['title'] = "<span style='font-size: 16px;'><b>State<b></span>"
fig['layout']['yaxis']['title'] = "<span style='font-size: 16px;'><b>Femur circumference (mm)<b></span>"

fig.update_layout(showlegend=False)
fig.update_xaxes(tickfont=dict(size=14))
fig.update_yaxes(tickfont=dict(size=14))
fig.show()

In [ ]:
# Femur_length_mm according to State
# Show the statistical parameters per state

femur_circumference_state=df.groupby("State")['Femur circumference (mm)']
femur_circumference_state.describe()

In [ ]:
# Show pairplots variables colored according to state

pairplot_1=sns.pairplot(data, hue='State',
             x_vars=['Femur length (mm)',
                     'Femur circumference (mm)',
                     'Body mass (g)'],
             y_vars=['Femur length (mm)',
                     'Femur circumference (mm)',
                     'Body mass (g)'],
             kind= "scatter",
             palette= "Paired",
             height=3,
             corner=True)

sns.move_legend(pairplot_1, "center", bbox_to_anchor=(0.7, 0.7))

plt.rc('axes', labelsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)

plt.setp(pairplot_1._legend.get_title(), fontsize=14)
plt.setp(pairplot_1._legend.get_texts(), fontsize=12)

# Print and save as .jpeg file
plt.savefig('Figura8_1.jpeg',dpi=500,bbox_inches="tight")

In [ ]:
#Create a copy of the dataset

data_eda = data.copy()

In [ ]:
# Change columns to logarithmic scale

data_eda['Femur length (mm)'] = data_eda['Femur length (mm)'].apply(np.log10)
data_eda['Femur circumference (mm)'] = data_eda['Femur circumference (mm)'].apply(np.log10)
data_eda['Body mass (g)'] = data_eda['Body mass (g)'].apply(np.log10)

#### Visualize data distribution after to change the variables to logarithmic scale

In [ ]:
# Show pairplots colored according to state

pairplot_2=sns.pairplot(data_eda,
                        hue='State',
                        x_vars=['Femur length (mm)',
                                'Femur circumference (mm)',
                                'Body mass (g)'],
                        y_vars=['Femur length (mm)',
                                'Femur circumference (mm)',
                                'Body mass (g)'],
                        kind= "scatter",
                        palette= "Paired",
                        height=3,
                        corner=True)

sns.move_legend(pairplot_2, "center", bbox_to_anchor=(0.7, 0.7))

plt.rc('axes', labelsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)

plt.setp(pairplot_2._legend.get_title(), fontsize=14)
plt.setp(pairplot_2._legend.get_texts(), fontsize=12)

plt.savefig('Figura8_2.jpeg',dpi=500,bbox_inches="tight")

In [ ]:
# Show the range of values per column and the statistical parameters per column

data_eda.describe()

In [ ]:
#Export describe as excel file

describe=data_eda.describe().to_excel("Table S5_2.xlsx", index=True)

In [ ]:
#Visual method- Normal distribution

sns.histplot(data_eda['Femur length (mm)'],
             color="cornflowerblue",
             kde=True)
plt.title('Femur length distribution', fontsize=14)
plt.rc('axes', labelsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.xlabel("Femur length (log)", fontsize=12)

# Print and save as .jpeg file
plt.savefig('Fl_histogram.jpeg',dpi=500,bbox_inches="tight")

In [ ]:
#Visual method- Normal distribution

sns.histplot(data_eda['Femur circumference (mm)'],
             color="cornflowerblue",
             kde=True)
plt.title('Femur circumference distribution', fontsize=14)
plt.rc('axes', labelsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.xlabel("Femur circumference (log)", fontsize=12)

# Print and save as .jpeg file
plt.savefig('Fc_histogram.jpeg',dpi=500,bbox_inches="tight")

## 4. Unsupervised learning - Cluster analysis
### K-Means

In [ ]:
#Create a copy of the dataset

data_log = data_eda.copy()

In [ ]:
data_log.drop(columns=['Higher clade', 'Species', 'State', 'Source', 'Body mass (g)'], inplace=True)

#### Choose the optimal number of clusters (K) for the analysis

In [ ]:
#The Silhouette Method

wcss=[]
for n_cluster in range(2, 11):
    km = KMeans(n_clusters=n_cluster, init='k-means++', random_state=5)
    preds = km.fit_predict(data_log)
    sil_coeff = silhouette_score(data_log, preds, metric='euclidean')
    print(f"For n_group={n_cluster}, The Silhouette Coefficient is {sil_coeff}")
    wcss.append(km.inertia_)

##### -This method suggests that the optimal number of clusters are 2, 3, 4 or 5.

In [ ]:
# The Elbow Method

plt.plot(range(2,11), wcss)
plt.title('The Elbow method', fontsize=14)
plt.xlabel('Number of clusters')
plt.ylabel('WCSS')   # within cluster sum of squares
plt.rc('axes', labelsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)

# Print and save as .jpeg file
plt.savefig('Figure 9_ElbowM.jpeg',dpi=500,bbox_inches="tight")

##### -This method suggests that the optimal number of clusters are  3 or 5.

In [ ]:
# Create K-means model

for k in range (2,6):
    kmeans = KMeans(n_clusters=k, random_state = 0)
    kmeans.fit(data_log)
    original_labels = kmeans.labels_
    centroids = kmeans.cluster_centers_
    sort_values = centroids.sum(axis=1)  # Group values are sorted based on centroids values
    sorted_indices = np.argsort(sort_values)
    label_mapping = {old: new for new, old in enumerate(sorted_indices)}
    sorted_labels = np.array([label_mapping[label] for label in original_labels])
    data[f'Groups_{k}'] = sorted_labels  #  Add each group column to the dataset called data

data.head(10)

In [ ]:
# This step add 1 unit to each group column's value

for column in data.columns:
    if column.startswith('Groups_'):
        data[column] = data[column] + 1
data.head()

In [ ]:
# Visualize Group Classification

fig,axs=plt.subplots(nrows=1,ncols=4,figsize=(20,5))
for i, ax in enumerate(fig.axes, start=2):
    ax.scatter (x=data['Body mass (g)'], y= data['Femur length (mm)'], c=data[f'Groups_{i}'],
                cmap='gist_rainbow')
    ax.set_xscale('log') # Set x-axis to logarithmic scale
    ax.set_yscale('log') # Set y-axis to logarithmic scale
    ax.set_xlabel('Body mass (g)')
    ax.set_ylabel('Femur length (mm)')
    ax.set_title(f'Groups:{i}')
plt.show()

In [ ]:
#Create a copy of the dataset

data_pairplot = data.copy()

In [ ]:
# Change columns to logarithmic scale

data_pairplot['Femur length (mm)'] = data_pairplot['Femur length (mm)'].apply(np.log10)
data_pairplot['Femur circumference (mm)'] = data_pairplot['Femur circumference (mm)'].apply(np.log10)
data_pairplot['Body mass (g)'] = data_pairplot['Body mass (g)'].apply(np.log10)

In [ ]:
# Show pairplots colored according to Groups

pairplot_3=sns.pairplot(data_pairplot,
             hue='Groups_3',
             x_vars=['Femur length (mm)',
                     'Femur circumference (mm)',
                     'Body mass (g)'],
             y_vars=['Femur length (mm)',
                     'Femur circumference (mm)',
                     'Body mass (g)'],
             kind= "scatter",
             palette= "Paired",
             height=3,
             corner=True
             )

sns.move_legend(pairplot_3, "center", bbox_to_anchor=(0.7, 0.7))

plt.rc('axes', labelsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)

plt.setp(pairplot_3._legend.get_title(), fontsize=14)
plt.setp(pairplot_3._legend.get_texts(), fontsize=12)

plt.savefig('Figure9_pairplt.jpeg',dpi=500,bbox_inches="tight")

In [ ]:
#Convert color_dict to matplotlib format

def rgba_to_mpl(rgba_str):
    values = rgba_str.replace("rgba(", "").replace(")", "").split(",")
    r, g, b, a = map(float, values)
    return (r/255, g/255, b/255, a)

color_dict_mpl = {k: rgba_to_mpl(v) for k, v in color_dict.items()}

In [ ]:
# Group 3 according to Higher clade
# Compute frequencies

freq = data.groupby(['Higher clade', 'Groups_3']).size().reset_index(name='count')
freq['color'] = freq['Higher clade'].map(color_dict_mpl)

# Convert categories to numeric positions
clade_codes = {clade: i for i, clade in enumerate(freq['Higher clade'].unique())}
freq['clade_num'] = freq['Higher clade'].map(clade_codes)

# Plot
plt.figure(figsize=(14, 6))

scatter = plt.scatter(
    x=freq['clade_num'],
    y=freq['Groups_3'],
    s=freq['count'] * 25, # scale bubble size
    c=freq['color'],
    alpha=0.5
)

# Axis formatting
plt.xticks(ticks=list(clade_codes.values()), labels=list(clade_codes.keys()), rotation=90)
plt.yticks([0, 1, 2, 3, 4])
plt.xlabel("Higher Clade")
plt.ylabel("Clusters: 3")
plt.title("Cluster Frequency per Higher Clade")
plt.tight_layout()
plt.show()

In [ ]:
# Show the statistical parameters of Body Mass per group

bodymass_3=data.groupby("Groups_3")['Body mass (g)']
bodymass_3.describe()

In [ ]:
#Export describe as excel file

table_1=bodymass_3.describe().to_excel("Table S_1.xlsx", index=True)

In [ ]:
# Show the statistical parameters of Femur Lenght per group

flenght_3=data.groupby("Groups_3")['Femur length (mm)']
flenght_3.describe()

In [ ]:
#Export describe as excel file

table_2=flenght_3.describe().to_excel("Table S_2.xlsx", index=True)

In [ ]:
# Show the statistical parameters of Femur Circumference per group

fcircunf_3=data.groupby("Groups_3")['Femur circumference (mm)']
fcircunf_3.describe()

In [ ]:
#Export describe as excel file
table_3=fcircunf_3.describe().to_excel("Table S_3.xlsx", index=True)

--------------------------------------------------------------------------------


In [ ]:
# Open dataset with colab

from google.colab import files
uploaded = files.upload()
data_error = pd.read_excel("error_data.xlsx")
data_error

In [ ]:
# Femur Circumference to higher clade

fig = px.box(data_error,
             x='Higher clade',
             y='PPE',
             color= 'Higher clade',
             color_discrete_map=color_dict,
             points="outliers",
             title=("<span style='font-size: 18px;'><b>PPE according to Higher clade<b></span>"
             ),
             width=1100,
             height=600)

fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor= "lightgray"
)

fig['layout']['xaxis']['title'] = "<span style='font-size: 16px;'><b>Higher clade<b></span>"
fig['layout']['yaxis']['title'] = "<span style='font-size: 16px;'><b>PPE<b></span>"

fig.update_layout(showlegend=False)
fig.update_xaxes(tickfont=dict(size=14))
fig.update_yaxes(tickfont=dict(size=14))
fig.show()

In [ ]:
# Femur Circumference to higher clade

fig = px.box(data_error,
             x='State',
             y='PPE',
             color= 'State',
             points="outliers",
             title=("<span style='font-size: 18px;'><b>PPE according to State<b></span>"
             ),
             width=500,
             height=500)

fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor= "lightgray"
)

fig['layout']['xaxis']['title'] = "<span style='font-size: 16px;'><b>State<b></span>"
fig['layout']['yaxis']['title'] = "<span style='font-size: 16px;'><b>PPE<b></span>"

fig.update_layout(showlegend=False)
fig.update_xaxes(tickfont=dict(size=14))
fig.update_yaxes(tickfont=dict(size=14))
fig.show()

# End code